# 04장. 코사인 유사도와 식단 군집

| 핵심 질문 | 학습 시간 |
|---|---:|
| 비슷한 메뉴와 식단 묶음은 어떻게 찾을까? | 3회차 후반 · 약 90분 |


## 이 장에서 배울 내용

- 코사인 유사도를 벡터 방향의 가까움으로 설명할 수 있다.
- K-Means가 정답표 없이 중심을 옮기며 묶는 과정을 설명할 수 있다.
- 군집 이름이 건강 등급이 아니라 상대적 설명임을 구분할 수 있다.


## 생각 열기

두 메뉴에 비슷한 글자 조각이 많으면 숫자 화살표도 비슷한 방향을 가리킵니다. 방향이 가까운 메뉴를 찾고, 영양 수치의 모양이 비슷한 날짜끼리 묶으면 식단의 특징을 다른 관점에서 볼 수 있습니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **코사인 유사도** | 두 벡터의 방향이 얼마나 비슷한지 나타내는 값 |
| **군집** | 특징이 비슷해 한 묶음으로 분류된 데이터 |
| **K-Means** | 가까운 중심을 찾아 반복해서 묶는 알고리즘 |
| **표준화** | 단위가 다른 숫자를 비교 가능한 크기로 바꾸는 일 |


## 개념 익히기


두 학생이 같은 방향을 바라보면 키가 달라도 방향은 같습니다. 코사인 유사도는 벡터의 길이보다 방향을 비교합니다.

K-Means는 운동장에 몇 개의 깃발을 놓고 학생들이 가장 가까운 깃발로 모이는 모습을 떠올리면 됩니다. 모인 학생들의 가운데로 깃발을 옮기고 다시 모으기를 반복합니다.

이 프로젝트의 군집은 열량·탄수화물·단백질·지방·메뉴 수의 상대 패턴입니다. ‘가벼운 구성’과 ‘든든한 구성’은 데이터 안의 비교 이름일 뿐 좋고 나쁨이 아닙니다.


## 활동 전 생각


점 A(1, 1), B(2, 2), C(1, 5)를 좌표에 찍어 보세요. A와 B는 같은 방향이지만 C는 다른 방향입니다. 코사인 유사도는 A와 B를 가깝게 봅니다.

K-Means 중심 이동도 숫자로 한 번 해 봅니다. A(1, 1), B(2, 2), C(8, 8), D(9, 9)가 있고 첫 중심을 A와 D에 놓습니다. 가까운 중심에 배정하면 A·B와 C·D로 나뉩니다. 첫 묶음의 새 중심은 `((1+2)/2, (1+2)/2)=(1.5, 1.5)`, 둘째 묶음은 `(8.5, 8.5)`로 이동합니다. 다시 배정해 묶음이 그대로면 반복을 멈춥니다.


## 예상하기

- 파스타 가상 취향의 1위는 파스타·피자가 있는 날짜다.
- 식단은 최소 두 종류의 상대 군집 이름으로 나뉜다.


## 활동 1. 유사도 1위 찾기


### 코드 살펴보기


1. `_tfidf_similarity`는 취향 문장과 각 메뉴의 유사도를 계산합니다.<br>
2. `scores.argmax()`는 가장 큰 유사도가 놓인 위치 번호를 찾습니다.<br>
3. `iloc[best_position]`은 그 위치의 메뉴 행을 꺼냅니다.<br>
4. `round(..., 3)`은 원래 값을 바꾸지 않고 출력할 때만 셋째 자리로 반올림합니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. neis-meal-ai 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.recommender import _tfidf_similarity, cluster_meals

query = "파스타 피자 면"
scores = _tfidf_similarity(meal_df["menu_text"].tolist(), query)
best_position = int(scores.argmax())
top_similar_menu = meal_df.iloc[best_position]["menu_text"]
print("가상 취향:", query)
print("가장 비슷한 메뉴:", top_similar_menu)
print("유사도:", round(float(scores[best_position]), 3))


### 결과 해석하기

1위는 취향 글자와 가장 비슷한 메뉴입니다. 다른 취향을 넣으면 1위가 달라질 수 있습니다.


## 활동 2. K-Means 군집 결과 읽기


### 코드 살펴보기


`cluster_meals` 안에서는 다음 순서가 반복됩니다.<br>
1. 열량·탄수화물·단백질·지방·메뉴 수를 비슷한 크기로 표준화합니다.<br>
2. 임시 중심을 놓고 각 날짜를 가장 가까운 중심에 배정합니다.<br>
3. 묶인 날짜들의 평균 위치로 중심을 옮깁니다.<br>
4. 더는 묶음이 바뀌지 않으면 상대적인 설명 이름을 붙입니다. `set`은 중복 이름을 한 번씩만 남깁니다.


In [ ]:
clustered_df = cluster_meals(meal_df, max_clusters=3)
print(clustered_df[["date", "calories", "protein_g", "cluster_name"]].to_string(index=False))
cluster_names = sorted(set(clustered_df["cluster_name"]))
print("나타난 군집:", cluster_names)

chapter_result = {
    "chapter": "04",
    "top_similar_menu": top_similar_menu,
    "cluster_names": cluster_names,
}


### 결과 해석하기

군집명이 같으면 이번 5행의 영양 수치 패턴이 상대적으로 비슷하다는 뜻입니다. 학생 개인의 건강 상태는 입력하지 않았습니다.


## 탐구 활동

max_clusters를 2와 3으로 각각 실행하고 군집 이름과 날짜 묶음이 어떻게 달라지는지 기록하세요.

먼저 기본값으로 실행한 뒤 한 곳만 바꾸어 결과를 비교합니다.


In [ ]:
practice_cluster_count = 2
practice_clustered = cluster_meals(meal_df, max_clusters=practice_cluster_count)
print(practice_clustered[["date", "cluster_name"]].to_string(index=False))


### 관찰 기록

- 바꾼 것:  
- 달라진 결과:  
- 그렇게 된 까닭:


## 확인 문제

1. 코사인 유사도가 벡터의 무엇을 비교하나요?
2. K-Means에서 중심을 반복해서 옮기는 이유는 무엇인가요?
3. ‘상대적 가벼운 구성’을 건강한 메뉴라고 바꿔 말하면 안 되는 이유는 무엇인가요?


## 정답과 해설


1. 벡터의 방향을 비교합니다.<br>
2. 각 데이터와 가까운 묶음의 중심을 더 잘 찾기 위해서입니다.<br>
3. 작은 공개 데이터 안의 숫자 비교일 뿐 개인 건강 적합성을 판단하지 않았기 때문입니다.


## 핵심 정리

- 코사인 유사도는 숫자 벡터의 방향을 비교한다.
- K-Means는 가까운 중심으로 데이터를 반복해서 묶는다.
- 군집은 상대 패턴 설명이며 건강 등급이 아니다.

### 다음 장에서 배울 내용

05장에서는 유사도에 명시적인 보너스와 감점을 합쳐 설명 가능한 추천 점수를 만듭니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
